# Case 01 · Foundations — What *is* fine-tuning?

**Goal:** by the end you can explain, and *show in code*, why starting from a pre-trained model
beats starting from scratch — and you'll see exactly which weights change.

Pair this with `animation.html` (open it in a browser first). Runs on CPU in seconds.

---
### 60-second refresher
- **Training** = nudging weights downhill on a *loss* surface (`new = old − lr × gradient`).
- **Pre-training** = the expensive, once-off climb from random weights to a good general model.
- **Fine-tuning** = continue training that good model on *your* narrower task, with a **small** learning rate and **few** steps.
- The bet: a good starting point needs far less data/compute to reach a good answer.

In [ ]:
import torch, torch.nn as nn, math
import matplotlib.pyplot as plt
torch.manual_seed(0)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1 · A tiny model and two tasks

Our 'model' is a small MLP that maps `x → y`. A **task** is just a target curve `y = sin(x + phase)`.
Task A is `phase=0`. Later, Task B (`phase=1.2`) will 'arrive' as a new update — exactly the situation
in your self-driving question: the world shifted, retrain efficiently.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(1,64), nn.Tanh(),
                         nn.Linear(64,64), nn.Tanh(),
                         nn.Linear(64,1))

def task_data(phase, n=256):
    x = torch.linspace(-math.pi, math.pi, n).unsqueeze(1)
    return x, torch.sin(x + phase)

def train(model, x, y, steps, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.MSELoss()
    hist = []
    for _ in range(steps):
        opt.zero_grad(); loss = lossf(model(x), y); loss.backward(); opt.step()
        hist.append(loss.item())
    return hist

## 2 · Pre-train on Task A

Many steps from random weights → a model that fits `sin(x)` well.

In [ ]:
xa, ya = task_data(0.0)
model = make_model()
hist_a = train(model, xa, ya, steps=1500, lr=1e-2)

plt.figure(figsize=(6,3))
plt.plot(xa, ya, label='Task A target', lw=2)
with torch.no_grad(): plt.plot(xa, model(xa), '--', label='pretrained fit')
plt.legend(); plt.title(f'Pre-trained on Task A (final loss {hist_a[-1]:.5f})'); plt.show()

## 3 · A new task arrives — fine-tune vs train-from-scratch

Task B = `sin(x + 1.2)`. We give **both** approaches the *same* tiny budget (300 steps, small lr):
- **Fine-tune:** keep the pre-trained weights (warm start).
- **From scratch:** a fresh random model (cold start).

Watch which one wins on equal compute.

In [ ]:
import copy
xb, yb = task_data(1.2)

ft = copy.deepcopy(model)           # warm start = the pretrained weights
fresh = make_model()                # cold start = random

hist_ft     = train(ft,    xb, yb, steps=300, lr=1e-3)
hist_scratch= train(fresh, xb, yb, steps=300, lr=1e-3)

plt.figure(figsize=(6,3))
plt.plot(hist_ft, label=f'fine-tune (warm)  -> {hist_ft[-1]:.5f}')
plt.plot(hist_scratch, label=f'from scratch (cold) -> {hist_scratch[-1]:.5f}')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss (log)')
plt.legend(); plt.title('Same budget: warm start converges lower & faster'); plt.show()

## 4 · Animate the fit improving (the 'movie' of fine-tuning)

We re-run fine-tuning but snapshot the prediction every few steps and animate it.
If the animation doesn't render in your viewer, the saved `.gif`/inline HTML still works in Jupyter.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

snap_model = copy.deepcopy(model)
opt = torch.optim.Adam(snap_model.parameters(), lr=1e-3); lossf = nn.MSELoss()
frames = []
for s in range(120):
    opt.zero_grad(); l = lossf(snap_model(xb), yb); l.backward(); opt.step()
    if s % 3 == 0:
        with torch.no_grad(): frames.append((snap_model(xb).squeeze().clone(), l.item()))

fig, ax = plt.subplots(figsize=(6,3))
ax.plot(xb, yb, 'k', lw=2, label='Task B target')
(line,) = ax.plot(xb, frames[0][0], 'r--', lw=2, label='model')
ax.legend(loc='upper right')
def upd(i):
    line.set_ydata(frames[i][0]); ax.set_title(f'fine-tuning step {i*3}  loss={frames[i][1]:.4f}'); return (line,)
anim = animation.FuncAnimation(fig, upd, frames=len(frames), interval=120, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 5 · *Which* weights actually moved?

Fine-tuning changes **all** weights, but by *how much*? We measure `|after − before|` per layer.
This sets up Case 02: most of the useful change is low-rank, so we can fake it with tiny adapters (LoRA).

In [ ]:
before = make_model(); before.load_state_dict(model.state_dict())  # pretrained snapshot
after  = ft                                                          # after fine-tuning

for (n,p0),(_,p1) in zip(before.named_parameters(), after.named_parameters()):
    if p0.dim()==2:
        delta = (p1-p0).abs()
        print(f'{n:12s} shape {tuple(p0.shape)}  mean|Δw| {delta.mean():.4f}  max|Δw| {delta.max():.4f}')

# heatmap of the change in the first weight matrix
import torch
d = (after[0].weight - before[0].weight).abs().detach()
plt.figure(figsize=(5,3)); plt.imshow(d, aspect='auto', cmap='magma')
plt.colorbar(label='|Δw|'); plt.title('Layer-0 weight change during fine-tuning'); plt.show()

## 6 · Takeaways

1. **Warm start wins** on equal compute — the core economic argument for fine-tuning.
2. Fine-tuning uses a **small lr / few steps** so you adapt without destroying the pre-trained knowledge.
3. The weight change is structured — that's the door to **parameter-efficient** methods (LoRA, Case 02).

### Connect to your capstone (self-driving)
- 'Task B arrives' = a new data batch / new city / new sensor. Fine-tuning is how you adapt cheaply.
- But notice: after fine-tuning on B, did the model get *worse* on A? We didn't check... that gap is
  **catastrophic forgetting**, the villain of Case 05. Hold that thought.

➡️ Now do `challenge.md`, then move to `cases/02_lora_from_scratch/`.